# Import Dependencies

In [ ]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

# Dataset Load

In [ ]:
train_df = pd.read_csv("datasets/CICIOMT/train/ciciomt2024_train_balanced.csv")
val_df = pd.read_csv("datasets/CICIOMT/val/ciciomt2024_validation_balanced.csv")
test_df = pd.read_csv("datasets/CICIOMT/test/ciciomt2024_test_balanced.csv")

# Define feature columns and target label

In [ ]:
non_feature_cols = [
    "source_file",
    "attack_type",
    "binary_label"
]

feature_cols = [col for col in train_df.columns if col not in non_feature_cols]
target_label = "binary_label"

# Train, Validation and Test

In [ ]:
# Creating features for training
X_train = train_df[feature_cols]
y_train = train_df[target_label]

# Creating features for validation
X_val = val_df[feature_cols]
y_val = val_df[target_label]

# Creating features for test
X_test = test_df[feature_cols]
y_test = test_df[target_label]

# Small Samples from Test Data

In [ ]:
X_shap = X_test.sample(n=5000, random_state=42)
print(X_shap.shape)

# SHAP for Full XGBoost Model

## XGBoost Training

In [ ]:
# XGBoost model
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

# train
xgb.fit(X_train, y_train)

# Get feature importance from trained XGBoost model
importance_values = xgb.feature_importances_

# Create Dataframe
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importance_values
})

# Sort by importance
importance_df = importance_df.sort_values(by="importance", ascending=False)

# SHAP Summary for Full Model

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_shap)

shap_importance = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    by="mean_abs_shap",
    ascending=False
)

print(shap_importance.head(20))

shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
plt.tight_layout()
plt.show()

shap.summary_plot(shap_values, X_shap, show=False)
plt.tight_layout()
plt.show()

# SHAP for Top-20 XGBoost

In [ ]:
top20_features = importance_df.head(20)["feature"].tolist()

print("Top-20 features:")
print(top20_features)


X_train_top20 = X_train[top20_features]
X_val_top20 = X_val[top20_features]
X_test_top20 = X_test[top20_features]

print("X_train_top20:", X_train_top20.shape)
print("X_val_top20:", X_val_top20.shape)
print("X_test_top20:", X_test_top20.shape)

X_shap_top20 = X_test_top20.sample(n=5000, random_state=42)

print("SHAP sample shape:", X_shap_top20.shape)

explainer_top20 = shap.TreeExplainer(xgb_top20)
shap_values_top20 = explainer_top20.shap_values(X_shap_top20)

shap_importance_top20 = pd.DataFrame({
    "feature": X_shap_top20.columns,
    "mean_abs_shap": np.abs(shap_values_top20).mean(axis=0)
})

shap_importance_top20 = shap_importance_top20.sort_values(
    by="mean_abs_shap",
    ascending=False
)

print("\nTop-20 Model SHAP Feature Importance:")
print(shap_importance_top20)

plt.figure()
shap.summary_plot(
    shap_values_top20,
    X_shap_top20,
    plot_type="bar",
    show=False
)
plt.tight_layout()
plt.show()

plt.figure()
shap.summary_plot(
    shap_values_top20,
    X_shap_top20,
    show=False
)
plt.tight_layout()
plt.show()

# Top-10 SHAP overlap score

In [ ]:
# Get top 10 features from both SHAP tables
full_top10 = shap_importance.head(10)["feature"].tolist()
top20_top10 = shap_importance_top20.head(10)["feature"].tolist()

print("Full model SHAP Top-10:")
print(full_top10)

print("\nTop-20 model SHAP Top-10:")
print(top20_top10)

# Find common features
common_features = set(full_top10).intersection(set(top20_top10))

print("\nCommon features:")
print(common_features)

# Calculate overlap
overlap_score = len(common_features) / 10

print("\nTop-10 SHAP Overlap Score:", overlap_score)

overlap_df = pd.DataFrame({
    "comparison": ["Full XGBoost vs Top-20 XGBoost"],
    "top_k": [10],
    "common_features": [", ".join(common_features)],
    "overlap_score": [overlap_score]
})

print(overlap_df)